In [1]:
import openpyxl
from sklearn.model_selection import train_test_split
import string
from pathlib import Path
import pandas as pd

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [2]:
path_to_images = Path('../plantvillage-dataset/raw/color/')

image_paths = [item for item in path_to_images.iterdir() if item.is_dir()]
filtered_classes = ['Blueberry', 'Corn', 'Grape', 'Orange', 'Raspberry', 'Soybean', 'Squash']

image_paths = [item for item in image_paths if not any(word in str(item) for word in filtered_classes)]
image_paths.sort()

In [53]:
path_to_labels = Path('../plantvillage-dataset/leaf_grouping/leaf_maps/')
label_paths = [item for item in path_to_labels.iterdir() if item.is_file()]
label_paths = [item for item in label_paths if not ('.csv' in str(item))]

filtered_labels = ['Blueberry', 'Grape', 'Soy', 'Citrus', 'Raspberry']
label_paths = [item for item in label_paths if not any(word in str(item) for word in filtered_labels)]

# Updating Sheet name for NREC Peppers and YLCV Tomato
pepper_idx = next((i for i, x in enumerate(label_paths) if ('Pepper' in str(x))), None)
pepper_wb = openpyxl.load_workbook(label_paths[pepper_idx])
old_name = 'Bac spot NREC'
new_name = 'Bacterial Spot Nrec'

try:
    pepper_sheet = pepper_wb[old_name]
    pepper_sheet.title = new_name

except KeyError:
    pass

old_name = 'Bacterial Spot JR'
new_name = 'Bacterial Spot Jr'

try:
    pepper_sheet = pepper_wb[old_name]
    pepper_sheet.title = new_name

except KeyError:
    pass

pepper_wb.save(label_paths[pepper_idx])
pepper_wb.close()

tomato_idx = next((i for i, x in enumerate(label_paths) if ('Tomato' in str(x))), None)
tomato_wb = openpyxl.load_workbook(label_paths[tomato_idx])
old_name = 'YLCV Virus'
new_name = 'Tomato Yellow Leaf Curl Virus'

try:
    tomato_sheet = tomato_wb[old_name]
    tomato_sheet.title = new_name

except KeyError:
    pass

tomato_wb.save(label_paths[tomato_idx])
tomato_wb.close()

In [7]:
apple_image_path = [path for path in image_paths if 'Apple' in str(path)]
apple_label_path = [path for path in label_paths if 'Apple' in str(path)]

In [40]:
records = []

for images in apple_image_path:
    class_name, subclass_name = str(images).split('/')[-1].split('___')
    class_name = class_name.split("_")[0].split(",")[0]
    subclass_name = string.capwords(subclass_name.replace("_", " "))
    path_to_class_labels = next((i for i in label_paths if (class_name in str(i))), None)

    excel_structure = pd.ExcelFile(path_to_class_labels)
    sheets_in_file = excel_structure.sheet_names

    sheet_name = next((sheet for sheet in sheets_in_file
                       if (sheet in subclass_name)
                       or (subclass_name in sheet)), None)

    if sheet_name is None:
        print(f"No labels for subclass {subclass_name} found.")
        continue

    labels = pd.read_excel(path_to_class_labels, sheet_name=sheet_name)

    print(f"Working on {subclass_name}")
    for file_path in images.iterdir():
        if not file_path.is_file():
            print(f"File path {file_path} is not considered a file.")
            continue

        f = str(file_path)
        parts = f.split("___", 2)

        if len(parts) != 3:
            print(f"Skipping unexpected file name: {file_path}.")
            continue

        f_trunc = parts[-1]
        match = labels.loc[labels["File Name"] == f_trunc, "Leaf #"]

        if len(match) == 0:
            print(f"No label found for {f_trunc}.")
            continue

        if len(match) > 1:
            print(f"Multiple label found for {f_trunc}.")
            continue

        leaf_id = match.iloc[0]

        records.append({
            "Plant Name": class_name,
            "Subclass Name": subclass_name,
            "File Path": f,
            "Leaf ID": leaf_id
        })

df = pd.DataFrame(records)

Working on Apple Scab
No labels for subclass Black Rot found.
Working on Cedar Apple Rust
Working on Healthy


In [45]:
image_paths

[PosixPath('../plantvillage-dataset/raw/color/Apple___Apple_scab'),
 PosixPath('../plantvillage-dataset/raw/color/Apple___Black_rot'),
 PosixPath('../plantvillage-dataset/raw/color/Apple___Cedar_apple_rust'),
 PosixPath('../plantvillage-dataset/raw/color/Apple___healthy'),
 PosixPath('../plantvillage-dataset/raw/color/Cherry_(including_sour)___Powdery_mildew'),
 PosixPath('../plantvillage-dataset/raw/color/Cherry_(including_sour)___healthy'),
 PosixPath('../plantvillage-dataset/raw/color/Peach___Bacterial_spot'),
 PosixPath('../plantvillage-dataset/raw/color/Peach___healthy'),
 PosixPath('../plantvillage-dataset/raw/color/Pepper,_bell___Bacterial_spot_JR'),
 PosixPath('../plantvillage-dataset/raw/color/Pepper,_bell___Bacterial_spot_NREC'),
 PosixPath('../plantvillage-dataset/raw/color/Pepper,_bell___healthy'),
 PosixPath('../plantvillage-dataset/raw/color/Potato___Early_blight'),
 PosixPath('../plantvillage-dataset/raw/color/Potato___Late_blight'),
 PosixPath('../plantvillage-dataset/r

In [48]:
peppers_images = [path for path in image_paths if 'Pepper' in str(path)]
peppers_label = [path for path in label_paths if 'Pepper' in str(path)]

In [49]:
peppers_images

[PosixPath('../plantvillage-dataset/raw/color/Pepper,_bell___Bacterial_spot_JR'),
 PosixPath('../plantvillage-dataset/raw/color/Pepper,_bell___Bacterial_spot_NREC'),
 PosixPath('../plantvillage-dataset/raw/color/Pepper,_bell___healthy')]

In [54]:
test = peppers_images[0]
class_name, subclass_name = str(test).split('/')[-1].split('___')
class_name = class_name.split("_")[0].split(",")[0]
subclass_name = string.capwords(subclass_name.replace("_", " "))

In [55]:
print(f"Class: {class_name}\nSubclass: {subclass_name}")

Class: Pepper
Subclass: Bacterial Spot Jr
